# Dopredné vs. spätné mapovanie pri geometrických transformáciách obrazu

Notebook oddeľuje dve úlohy pri prevzorkovaní:

- homogénna matica $M$ pôsobí na **polohy** $(x,y,1)^T$;
- bilineárne jadro $h$ kombinuje **hodnoty vzoriek** $f[k]$.

Pre každú polohu výstupnej mriežky $p\in\mathbb Z^2$ počítame

$$g(p)=\widetilde f(M^{-1}p),\qquad
\widetilde f(x)=\sum_k f[k]h(x-k).$$

Obrázok je zámerne malý, aby boli obe mriežky aj všetky interpolačné váhy viditeľné.

In [1]:
import colorsys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "font.size": 9,
})

H = W = 10
CENTRE = np.array([(W - 1) / 2, (H - 1) / 2])


## Transformácia a vzorkovanie

Polohy zapisujeme vždy ako **$(x,y)$**. Polia NumPy sa indexujú **`[riadok, stĺpec] = [y, x]`**. Je to jediný prevod súradníc, ktorý nižšie potrebujeme.

In [2]:
def make_M(phi_deg, s, c=CENTRE):
    # Vráti T(c) @ R(phi) @ S(s) @ T(-c).
    phi = np.deg2rad(phi_deg)
    co, si = np.cos(phi), np.sin(phi)
    T_to_origin = np.array([[1, 0, -c[0]], [0, 1, -c[1]], [0, 0, 1.0]])
    S = np.array([[s, 0, 0], [0, s, 0], [0, 0, 1.0]])
    R = np.array([[co, -si, 0], [si, co, 0], [0, 0, 1.0]])
    T_back = np.array([[1, 0, c[0]], [0, 1, c[1]], [0, 0, 1.0]])
    return T_back @ R @ S @ T_to_origin


def input_colours(height=H, width=W):
    # Vzorky hladkého poľa HLS: odtieň sa mení s x, svetlosť s y.
    img = np.empty((height, width, 3), dtype=float)
    for y in range(height):
        for x in range(width):
            hue = 0.72 * x / max(width - 1, 1)
            lightness = 0.30 + 0.45 * y / max(height - 1, 1)
            img[y, x] = colorsys.hls_to_rgb(hue, lightness, 0.78)
    return img


def bilinear_weights(x, y):
    # Vráti štyroch celočíselných susedov (x, y) a ich bilineárne váhy.
    x0, y0 = int(np.floor(x)), int(np.floor(y))
    dx, dy = x - x0, y - y0
    neighbours = np.array([[x0, y0], [x0 + 1, y0],
                           [x0, y0 + 1], [x0 + 1, y0 + 1]])
    weights = np.array([(1 - dx) * (1 - dy), dx * (1 - dy),
                        (1 - dx) * dy, dx * dy])
    return neighbours, weights


def backward_warp(img, M, output_shape=None):
    # Vyhodnotí g(p) = f~(inv(M)p) na výstupnej celočíselnej mriežke.
    out_h, out_w = output_shape or img.shape[:2]
    yy, xx = np.meshgrid(np.arange(out_h), np.arange(out_w), indexing="ij")
    pts = np.vstack((xx.ravel(), yy.ravel(), np.ones(xx.size)))
    src = np.linalg.inv(M) @ pts
    x, y = src[0], src[1]

    valid = (x >= 0) & (x <= img.shape[1] - 1) & (y >= 0) & (y <= img.shape[0] - 1)
    x0 = np.floor(x).astype(int); y0 = np.floor(y).astype(int)
    x1 = np.clip(x0 + 1, 0, img.shape[1] - 1)
    y1 = np.clip(y0 + 1, 0, img.shape[0] - 1)
    x0 = np.clip(x0, 0, img.shape[1] - 1)
    y0 = np.clip(y0, 0, img.shape[0] - 1)
    dx, dy = x - np.floor(x), y - np.floor(y)

    values = ((1-dx)[:, None] * (1-dy)[:, None] * img[y0, x0]
              + dx[:, None] * (1-dy)[:, None] * img[y0, x1]
              + (1-dx)[:, None] * dy[:, None] * img[y1, x0]
              + dx[:, None] * dy[:, None] * img[y1, x1])
    values[~valid] = np.nan
    return values.reshape(out_h, out_w, -1), valid.reshape(out_h, out_w), src[:2].T.reshape(out_h, out_w, 2)


IMG = input_colours()


Vykreslenie

In [3]:
def lattice(width=W, height=H):
    yy, xx = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
    return np.column_stack((xx.ravel(), yy.ravel()))


def transform_points(points, M):
    hom = np.column_stack((points, np.ones(len(points))))
    mapped = (M @ hom.T).T
    return mapped[:, :2] / mapped[:, 2, None]


def style_axis(ax, xlim, ylim):
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(*xlim)
    ax.set_ylim(ylim[1], ylim[0])  # konvencia obrazu: y rastie smerom nadol
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(False)


def common_limits(a, b, margin=0.7):
    both = np.vstack((a, b))
    return ((both[:, 0].min() - margin, both[:, 0].max() + margin),
            (both[:, 1].min() - margin, both[:, 1].max() + margin))


def plot_forward(ax, img, M):
    inp = lattice(img.shape[1], img.shape[0])
    out = lattice(img.shape[1], img.shape[0])
    moved = transform_points(inp, M)
    ax.scatter(out[:, 0], out[:, 1], marker="x", s=20, linewidths=0.8,
               color="0.58", label=r"výstupná mriežka $\mathbb{Z}^2$")
    ax.scatter(moved[:, 0], moved[:, 1], s=44, c=img.reshape(-1, 3),
               edgecolors="white", linewidths=0.35, zorder=3, label=r"dopredné vzorky $Mk$")
    xlim, ylim = common_limits(out, moved)
    style_axis(ax, xlim, ylim)
    ax.set_title(r"Dopredné: $M(\mathbb{Z}^2)$ vo výstupnej sústave")
    ax.legend(loc="upper right", fontsize=7, framealpha=0.9)


def plot_backward(ax, img, M, highlight):
    inp = lattice(img.shape[1], img.shape[0])
    out = lattice(img.shape[1], img.shape[0])
    pulled = transform_points(out, np.linalg.inv(M))
    ax.scatter(inp[:, 0], inp[:, 1], s=44, c=img.reshape(-1, 3),
               edgecolors="white", linewidths=0.35, zorder=2, label="vstupné vzorky $k$")
    ax.scatter(pulled[:, 0], pulled[:, 1], s=11, color="black", alpha=0.70,
               zorder=3, label=r"spätne zobrazené $M^{-1}p$")

    q = transform_points(np.array([highlight]), np.linalg.inv(M))[0]
    neighbours, weights = bilinear_weights(*q)
    ax.scatter(q[0], q[1], s=105, facecolor="gold", edgecolor="black", zorder=6)
    for (nx, ny), weight in zip(neighbours, weights):
        ax.plot([q[0], nx], [q[1], ny], color="0.2", lw=0.75, zorder=4)
        mx, my = (q[0] + nx) / 2, (q[1] + ny) / 2
        label_offset = (-30 if nx < q[0] else 8, -18 if ny < q[1] else 8)
        ax.annotate(f"w={weight:.3f}", (mx, my), xytext=label_offset,
                    textcoords="offset points", fontsize=7,
                    bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.78), zorder=7)
    ax.annotate(fr"$p=({highlight[0]},{highlight[1]})$" + "\n" +
                fr"$M^{{-1}}p=({q[0]:.2f},{q[1]:.2f})$",
                q, xytext=(12, -48), textcoords="offset points", fontsize=7,
                bbox=dict(boxstyle="round", fc="white", ec="0.5", alpha=0.92), zorder=8)
    xlim, ylim = common_limits(inp, pulled)
    style_axis(ax, xlim, ylim)
    ax.set_title(r"Spätné: $M^{-1}(\mathbb{Z}^2)$ vo vstupnej sústave")
    ax.legend(loc="upper right", fontsize=7, framealpha=0.9)
    return weights


def plot_result(ax, warped, valid, highlight):
    out = lattice(warped.shape[1], warped.shape[0])
    flat_valid = valid.ravel()
    ax.scatter(out[~flat_valid, 0], out[~flat_valid, 1], s=38, facecolors="none",
               edgecolors="0.55", linewidths=0.9, label="nedefinované")
    ax.scatter(out[flat_valid, 0], out[flat_valid, 1], s=58,
               c=warped.reshape(-1, 3)[flat_valid], edgecolors="white", linewidths=0.4)
    hx, hy = highlight
    ax.scatter(hx, hy, s=135, facecolors="none", edgecolors="black", linewidths=1.8, zorder=5)
    style_axis(ax, (-0.7, warped.shape[1] - 0.3), (-0.7, warped.shape[0] - 0.3))
    ax.set_title(r"Výsledok $g$ na $\mathbb{Z}^2$")
    ax.legend(loc="upper right", fontsize=7, framealpha=0.9)


def zero_weight_count(img, M):
    # Spočíta vstupné vzorky s nulovou celkovou bilineárnou váhou.
    _, valid, src = backward_warp(img, M)
    totals = np.zeros(img.shape[:2], dtype=float)
    for x, y in src[valid]:
        neighbours, weights = bilinear_weights(x, y)
        for (nx, ny), weight in zip(neighbours, weights):
            if 0 <= nx < img.shape[1] and 0 <= ny < img.shape[0]:
                totals[ny, nx] += weight
    return int(np.count_nonzero(totals <= 1e-14)), totals


def make_figure(phi=25, s=1.6, highlight=(5, 5), alias_note=None):
    if alias_note is None:
        alias_note = s < 1
    M = make_M(phi, s)
    warped, valid, _ = backward_warp(IMG, M)
    fig, axes = plt.subplots(1, 3, figsize=(15.2, 5.15), constrained_layout=False)
    plot_forward(axes[0], IMG, M)
    weights = plot_backward(axes[1], IMG, M, highlight)
    plot_result(axes[2], warped, valid, highlight)
    fig.suptitle(fr"Rotácia $\phi={phi:.1f}^\circ$, mierka $s={s:.2f}$", y=0.98, fontsize=13)
    fig.text(0.5, 0.035,
             r"Panely 1 a 2 ukazujú tú istú dvojicu mriežok – vo výstupnej a vo vstupnej sústave; prechod medzi nimi je $M$.",
             ha="center", fontsize=9)
    if alias_note:
        unread, _ = zero_weight_count(IMG, M)
        fig.text(0.5, 0.075,
                 f"Aliasing pri zmenšení: spätne zobrazená výstupná mriežka je riedka; "
                 f"{unread}/{IMG.shape[0] * IMG.shape[1]} vstupných vzoriek má nulovú celkovú váhu.",
                 ha="center", color="firebrick", fontsize=9.5,
                 bbox=dict(boxstyle="round,pad=0.3", fc="#fff5f2", ec="firebrick", alpha=0.95))
    fig.subplots_adjust(left=0.045, right=0.985, top=0.88, bottom=0.17, wspace=0.20)
    return fig, axes, weights


## Interaktívne skúmanie

Posuvník bodu indexuje výstupnú mriežku po riadkoch: index $i$ znamená $p=(i\bmod W,\lfloor i/W\rfloor)$. Ak je nainštalovaný `ipywidgets`, bunka použije widgety notebooku, inak otvorí posuvníky Matplotlibu.

Pri $s<1$ (zmenšenie) sa pod grafom zobrazí, koľko vstupných vzoriek nedostane žiadnu interpolačnú váhu.

In [5]:
def launch_interactive():
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output

        phi_w = widgets.FloatSlider(value=25, min=-60, max=60, step=1, description="uhol φ (°)")
        scale_w = widgets.FloatSlider(value=1.6, min=0.3, max=2.0, step=0.05, description="mierka s")
        point_w = widgets.IntSlider(value=55, min=0, max=H * W - 1, step=1, description="index bodu")
        output = widgets.Output()

        def redraw(*_):
            with output:
                clear_output(wait=True)
                p = (point_w.value % W, point_w.value // W)
                fig, _, _ = make_figure(phi_w.value, scale_w.value, p)
                display(fig)
                plt.close(fig)

        for widget in (phi_w, scale_w, point_w):
            widget.observe(redraw, names="value")
        display(widgets.VBox([widgets.HBox([phi_w, scale_w, point_w]), output]))
        redraw()
    except ImportError:
        from matplotlib.widgets import Slider

        fig = plt.figure(figsize=(15.2, 6.25))
        grid = fig.add_gridspec(1, 3, left=0.05, right=0.985, top=0.86, bottom=0.24, wspace=0.20)
        axes = [fig.add_subplot(grid[0, i]) for i in range(3)]
        phi_ax = fig.add_axes([0.12, 0.15, 0.22, 0.025])
        scale_ax = fig.add_axes([0.42, 0.15, 0.22, 0.025])
        point_ax = fig.add_axes([0.72, 0.15, 0.20, 0.025])
        phi_s = Slider(phi_ax, "uhol φ (°)", -60, 60, valinit=25, valstep=1)
        scale_s = Slider(scale_ax, "mierka s", 0.3, 2.0, valinit=1.6, valstep=0.05)
        point_s = Slider(point_ax, "index bodu", 0, H * W - 1, valinit=55, valstep=1)

        def redraw(_=None):
            for ax in axes:
                ax.clear()
            idx = int(point_s.val)
            p = (idx % W, idx // W)
            M = make_M(phi_s.val, scale_s.val)
            warped, valid, _ = backward_warp(IMG, M)
            plot_forward(axes[0], IMG, M)
            plot_backward(axes[1], IMG, M, p)
            plot_result(axes[2], warped, valid, p)
            fig.suptitle(fr"Rotácia $\phi={phi_s.val:.1f}^\circ$, mierka $s={scale_s.val:.2f}$", y=0.97)
            for t in list(fig.texts):
                t.remove()
            fig.text(0.5, 0.045,
                     r"Panely 1 a 2 ukazujú tú istú dvojicu mriežok – vo výstupnej a vo vstupnej sústave; prechod medzi nimi je $M$.",
                     ha="center", fontsize=9)
            if scale_s.val < 1:
                unread, _ = zero_weight_count(IMG, M)
                fig.text(0.5, 0.09,
                         f"Aliasing pri zmenšení: {unread}/{H * W} vstupných vzoriek má nulovú celkovú váhu.",
                         ha="center", color="firebrick", fontsize=9.5)
            fig.canvas.draw_idle()

        for slider in (phi_s, scale_s, point_s):
            slider.on_changed(redraw)
        redraw()
        plt.show()
        return fig


# Spustite v živej relácii notebooku, zobrazia sa ovládacie prvky:
launch_interactive()


## Prečo prevzorkovanie používa spätné mapovanie?

